In [ ]:
import pandas as pd
import numpy as np
import zipfile
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
# ─────────────────────────────────────────────────────────────
# ① 데이터 로딩 (eda.ipynb와 동일한 방식)
# ─────────────────────────────────────────────────────────────
data_dir = "data"

zip_files = {
    "2507": "카드소비 데이터_202507.zip",
    "2508": "카드소비 데이터_202508.zip",
    "2509": "카드소비 데이터_202509.zip",
    "2510": "카드소비 데이터_202510.zip",
    "2511": "카드소비 데이터_202511.zip",
    "2512": "카드소비 데이터_202512.zip",
}

all_dfs = []
for month, zipname in zip_files.items():
    zip_path = os.path.join(data_dir, zipname)
    monthly_dfs = []
    with zipfile.ZipFile(zip_path) as zf:
        for filename in sorted(zf.namelist()):
            with zf.open(filename) as f:
                df_city = pd.read_csv(f)
                city_name = filename.split("_")[-1].replace(".csv", "")
                df_city["시군구"] = city_name
                monthly_dfs.append(df_city)
    combined = pd.concat(monthly_dfs, ignore_index=True)
    combined['month'] = month
    all_dfs.append(combined)
    print(f"✅ {month} 로드 완료 → shape: {combined.shape}")

# ─────────────────────────────────────────────────────────────
# ② 전체 통합 & 전처리
# ─────────────────────────────────────────────────────────────
df_all = pd.concat(all_dfs, ignore_index=True)
print(f"\n✅ 전체 합치기 완료 → shape: {df_all.shape}")

df_all = df_all.drop(columns=['시군구'], errors='ignore')
df_all = df_all[df_all['cnt'] > 0].copy()
df_all['건당가격'] = df_all['amt'] / df_all['cnt']

# IQR 이상치 제거 (card_tpbuz_nm_2 기준)
def remove_outliers_iqr(group):
    Q1 = group['건당가격'].quantile(0.25)
    Q3 = group['건당가격'].quantile(0.75)
    IQR = Q3 - Q1
    return group[(group['건당가격'] >= Q1 - 1.5*IQR) & (group['건당가격'] <= Q3 + 1.5*IQR)]

before = len(df_all)
df_all = df_all.groupby('card_tpbuz_nm_2', group_keys=False).apply(remove_outliers_iqr)
print(f"   이상치 제거: {before:,} → {len(df_all):,}행")


In [ ]:
# ══════════════════════════════════════════════════════════════
# ③ 피처 엔지니어링: (age, sex) 단위 소비자 프로파일 집계
# ══════════════════════════════════════════════════════════════
base = df_all.groupby(['age', 'sex']).agg(
    평균_건당가격=('건당가격', 'mean'),
    총_거래건수=('cnt', 'sum'),
    총_결제금액=('amt', 'sum'),
    평균_시간대=('hour', 'mean'),
).reset_index()

cat_pivot = df_all.groupby(['age', 'sex', 'card_tpbuz_nm_1'])['amt'].sum().reset_index()
cat_pivot = cat_pivot.pivot_table(
    index=['age', 'sex'], columns='card_tpbuz_nm_1', values='amt', fill_value=0
)
cat_pivot = cat_pivot.div(cat_pivot.sum(axis=1), axis=0).round(4)
cat_pivot.columns = [f'업종비율_{c}' for c in cat_pivot.columns]
cat_pivot = cat_pivot.reset_index()

df_all['is_weekend'] = df_all['day'].apply(lambda x: 1 if x >= 5 else 0)
weekend = df_all.groupby(['age', 'sex'])['is_weekend'].mean().reset_index()
weekend.rename(columns={'is_weekend': '주말비율'}, inplace=True)

profile = base.merge(cat_pivot, on=['age', 'sex']).merge(weekend, on=['age', 'sex'])
feature_cols = [c for c in profile.columns if c not in ['age', 'sex']]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(profile[feature_cols].values)

print(f"\n소비자 프로파일 shape: {profile.shape}")


In [ ]:
# ══════════════════════════════════════════════════════════════
# ④ 최적 컴포넌트 수 탐색: BIC + AIC
#   (GMM은 Elbow 대신 BIC/AIC가 기준)
# ══════════════════════════════════════════════════════════════
K_range = range(2, 12)
bic_scores = []
aic_scores = []

print("\n[BIC / AIC 탐색]")
for k in K_range:
    gmm = GaussianMixture(n_components=k, random_state=42,
                          covariance_type='full', n_init=5, max_iter=300)
    gmm.fit(X_scaled)
    bic_scores.append(gmm.bic(X_scaled))
    aic_scores.append(gmm.aic(X_scaled))
    print(f"  k={k:2d} | BIC: {gmm.bic(X_scaled):10.1f} | AIC: {gmm.aic(X_scaled):10.1f}")

# 최적 K: BIC 최솟값
best_k = list(K_range)[bic_scores.index(min(bic_scores))]
print(f"\n★ BIC 기준 최적 K = {best_k}")

# ── BIC / AIC 시각화 ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(K_range), bic_scores, 'bo-', linewidth=2, markersize=7)
axes[0].axvline(x=best_k, color='orange', linestyle='--', linewidth=2,
                label=f'최적 K = {best_k}')
axes[0].set_title('BIC Score\n(낮을수록 좋음)', fontsize=13)
axes[0].set_xlabel('컴포넌트 수 K')
axes[0].set_ylabel('BIC')
axes[0].legend()
axes[0].grid(linestyle='--', alpha=0.6)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

axes[1].plot(list(K_range), aic_scores, 'rs-', linewidth=2, markersize=7)
axes[1].axvline(x=best_k, color='orange', linestyle='--', linewidth=2,
                label=f'최적 K = {best_k}')
axes[1].set_title('AIC Score\n(낮을수록 좋음)', fontsize=13)
axes[1].set_xlabel('컴포넌트 수 K')
axes[1].set_ylabel('AIC')
axes[1].legend()
axes[1].grid(linestyle='--', alpha=0.6)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.suptitle('GMM 최적 컴포넌트 수 탐색 (7월~12월 전체)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════
# ⑤ 최적 K로 GMM 학습
# ══════════════════════════════════════════════════════════════
gmm_final = GaussianMixture(n_components=best_k, random_state=42,
                             covariance_type='full', n_init=5, max_iter=300)
gmm_final.fit(X_scaled)

# 하드 배정 (가장 높은 확률 컴포넌트)
profile['cluster'] = gmm_final.predict(X_scaled)

# 소프트 배정 (각 컴포넌트 소속 확률)
proba_cols = [f'prob_C{i}' for i in range(best_k)]
proba_df = pd.DataFrame(
    gmm_final.predict_proba(X_scaled).round(3),
    columns=proba_cols
)
profile = pd.concat([profile.reset_index(drop=True), proba_df], axis=1)

print(f"\n[클러스터 분포 (하드 배정)]")
print(profile['cluster'].value_counts().sort_index())

print(f"\n[소프트 소속 확률 (상위 5행)]")
print(profile[['age', 'sex', 'cluster'] + proba_cols].head(10).to_string(index=False))

# ══════════════════════════════════════════════════════════════
# ⑥ 클러스터별 피처 평균 & 히트맵
# ══════════════════════════════════════════════════════════════
cluster_means = profile.groupby('cluster')[feature_cols].mean()

# 히트맵: GMM 평균 (표준화 기준)
means_scaled = pd.DataFrame(
    scaler.transform(cluster_means.values),
    columns=feature_cols,
    index=[f'Cluster {i}' for i in range(best_k)]
)

fig, ax = plt.subplots(figsize=(max(14, len(feature_cols) * 0.7), best_k * 1.2 + 2))
im = ax.imshow(means_scaled.values, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(len(feature_cols)))
ax.set_xticklabels(feature_cols, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(best_k))
ax.set_yticklabels([f'Cluster {i}' for i in range(best_k)], fontsize=11)
plt.colorbar(im, ax=ax, label='표준화된 값 (+ 높음 / - 낮음)')
for i in range(best_k):
    for j in range(len(feature_cols)):
        ax.text(j, i, f'{means_scaled.values[i, j]:.2f}',
                ha='center', va='center', fontsize=7)
ax.set_title('GMM 클러스터 평균 히트맵\n(초록=높음, 빨강=낮음)', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════
# ⑦ 소프트 배정 시각화: 각 그룹의 클러스터 소속 확률 스택바
# ══════════════════════════════════════════════════════════════
profile_sorted = profile.sort_values(['cluster', 'age', 'sex']).reset_index(drop=True)
labels = profile_sorted.apply(lambda r: f"age{r['age']}-{r['sex']}", axis=1)

fig, ax = plt.subplots(figsize=(16, 5))
palette = plt.cm.Set2.colors
bottom = np.zeros(len(profile_sorted))

for i in range(best_k):
    vals = profile_sorted[f'prob_C{i}'].values
    ax.bar(labels, vals, bottom=bottom,
           label=f'Cluster {i}',
           color=palette[i % len(palette)],
           edgecolor='white', linewidth=0.3)
    bottom += vals

ax.set_title('(age, sex) 그룹별 GMM 소속 확률 (소프트 배정)\n→ 한 그룹이 여러 클러스터에 걸쳐 있을 수 있음',
             fontsize=13)
ax.set_ylabel('소속 확률')
ax.set_ylim(0, 1)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════
# ⑧ 과소비 탐지 (클러스터 내 Q75 기준)
# ══════════════════════════════════════════════════════════════
def label_overspend(group):
    q75 = group['평균_건당가격'].quantile(0.75)
    group['과소비여부'] = group['평균_건당가격'] > q75
    return group

profile = profile.groupby('cluster', group_keys=False).apply(label_overspend)
print("\n[과소비군 탐지]")
print(profile[['age', 'sex', 'cluster', '평균_건당가격', '과소비여부']]
      .sort_values(['cluster', '평균_건당가격'], ascending=[True, False])
      .to_string(index=False))

# ══════════════════════════════════════════════════════════════
# ⑨ 개인 ID 데이터 예측 함수 (GMM 버전)
# ══════════════════════════════════════════════════════════════
CLUSTER_LABELS = {i: f'Cluster {i}' for i in range(best_k)}

def predict_individual_gmm(individual_df):
    """
    GMM 기반 개인 소비 유형 + 확률 + 과소비 판정
    Returns: dict with cluster probabilities (soft assignment)
    """
    df_ind = individual_df[individual_df['cnt'] > 0].copy()
    df_ind['건당가격'] = df_ind['amt'] / df_ind['cnt']

    feat = {
        '평균_건당가격': df_ind['건당가격'].mean(),
        '총_거래건수':   df_ind['cnt'].sum(),
        '총_결제금액':   df_ind['amt'].sum(),
        '평균_시간대':   df_ind['hour'].mean(),
        '주말비율':      (df_ind['day'] >= 5).mean(),
    }
    cat_amt = df_ind.groupby('card_tpbuz_nm_1')['amt'].sum()
    cat_ratio = (cat_amt / cat_amt.sum()).to_dict()
    for col in feature_cols:
        if col.startswith('업종비율_'):
            feat[col] = cat_ratio.get(col.replace('업종비율_', ''), 0.0)

    X_new = np.array([[feat.get(c, 0.0) for c in feature_cols]])
    X_new_scaled = scaler.transform(X_new)

    cluster_id    = int(gmm_final.predict(X_new_scaled)[0])
    probabilities = gmm_final.predict_proba(X_new_scaled)[0]

    # 과소비 판정
    q75 = profile[profile['cluster'] == cluster_id]['평균_건당가격'].quantile(0.75)
    is_overspend = feat['평균_건당가격'] > q75

    # 확률 상위 2개 뽑기
    top2_idx = probabilities.argsort()[::-1][:2]
    top2 = [(f"Cluster {i}", round(probabilities[i]*100, 1)) for i in top2_idx]

    return {
        'main_cluster':    cluster_id,
        'cluster_name':    CLUSTER_LABELS.get(cluster_id, f'Cluster {cluster_id}'),
        '건당가격':        round(feat['평균_건당가격']),
        '과소비여부':      is_overspend,
        '클러스터_Q75':    round(q75),
        '소속확률_top2':   top2,           # ← GMM의 핵심: 혼합 유형 표현
        '전체확률':        {f'C{i}': round(float(p)*100, 1) for i, p in enumerate(probabilities)},
    }

# ── 데모 실행 ─────────────────────────────────────────────────
demo_df = df_all[(df_all['age'] == 4) & (df_all['sex'] == 'F')]
result = predict_individual_gmm(demo_df)

print("\n" + "="*55)
print("  GMM 개인 소비 분석 결과 (40대 여성 데모)")
print("="*55)
print(f"  주 소비 유형  : {result['cluster_name']}")
print(f"  과소비 판정   : {'⚠️  과소비군' if result['과소비여부'] else '✅  일반소비군'}")
print(f"  건당 평균     : {result['건당가격']:,}원 (Q75: {result['클러스터_Q75']:,}원)")
print(f"  소속 확률 1위 : {result['소속확률_top2'][0][0]} {result['소속확률_top2'][0][1]}%")
print(f"  소속 확률 2위 : {result['소속확률_top2'][1][0]} {result['소속확률_top2'][1][1]}%")
print(f"  전체 확률     : {result['전체확률']}")
print("="*55)

In [ ]:
import pandas as pd
import numpy as np
import zipfile
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# ─────────────────────────────────────────────────────────────
# ① 데이터 로딩
# ─────────────────────────────────────────────────────────────
data_dir = "data"

zip_files = {
    "2507": "카드소비 데이터_202507.zip",
    "2508": "카드소비 데이터_202508.zip",
    "2509": "카드소비 데이터_202509.zip",
    "2510": "카드소비 데이터_202510.zip",
    "2511": "카드소비 데이터_202511.zip",
    "2512": "카드소비 데이터_202512.zip",
}

all_dfs = []
for month, zipname in zip_files.items():
    zip_path = os.path.join(data_dir, zipname)
    monthly_dfs = []
    with zipfile.ZipFile(zip_path) as zf:
        for filename in sorted(zf.namelist()):
            with zf.open(filename) as f:
                df_city = pd.read_csv(f)
                city_name = filename.split("_")[-1].replace(".csv", "")
                df_city["시군구"] = city_name
                monthly_dfs.append(df_city)
    combined = pd.concat(monthly_dfs, ignore_index=True)
    combined['month'] = month
    all_dfs.append(combined)
    print(f"✅ {month} 로드 완료 → shape: {combined.shape}")

df_all = pd.concat(all_dfs, ignore_index=True)
print(f"\n✅ 전체 합치기 완료 → shape: {df_all.shape}")

# ─────────────────────────────────────────────────────────────
# ② 전처리
# ─────────────────────────────────────────────────────────────
df_all = df_all.drop(columns=['시군구'], errors='ignore')
df_all = df_all[df_all['cnt'] > 0].copy()
df_all['건당가격'] = df_all['amt'] / df_all['cnt']

# 이상치 제거
def remove_outliers_iqr(group):
    Q1 = group['건당가격'].quantile(0.25)
    Q3 = group['건당가격'].quantile(0.75)
    IQR = Q3 - Q1
    return group[(group['건당가격'] >= Q1 - 1.5*IQR) & (group['건당가격'] <= Q3 + 1.5*IQR)]

before = len(df_all)
df_all = df_all.groupby('card_tpbuz_nm_2', group_keys=False).apply(remove_outliers_iqr)
print(f"   이상치 제거: {before:,} → {len(df_all):,}행")

# ─────────────────────────────────────────────────────────────
# ③ 대분류 확인
# ─────────────────────────────────────────────────────────────
cats = df_all['card_tpbuz_nm_1'].unique()
print(f"\n📦 대분류(card_tpbuz_nm_1) 종류 ({len(cats)}개):")
for c in sorted(cats):
    total_amt = df_all[df_all['card_tpbuz_nm_1'] == c]['amt'].sum()
    print(f"  - {c}: {total_amt:,.0f}원")

# ══════════════════════════════════════════════════════════════
# ④ 핵심: 대분류 업종 비율만으로 프로파일 생성
#    단위: (age, sex) 조합
#    피처: 각 대분류에서 지출한 비율 (합계=1)
# ══════════════════════════════════════════════════════════════
# (age, sex) × 대분류별 총 결제금액 피벗
cat_pivot = (
    df_all.groupby(['age', 'sex', 'card_tpbuz_nm_1'])['amt']
    .sum()
    .reset_index()
    .pivot_table(index=['age', 'sex'], columns='card_tpbuz_nm_1', values='amt', fill_value=0)
)

# 비율화 (행 합계 = 1)
cat_ratio = cat_pivot.div(cat_pivot.sum(axis=1), axis=0).round(4)
cat_ratio.columns = [f'비율_{c}' for c in cat_ratio.columns]
cat_ratio = cat_ratio.reset_index()

# 대분류 컬럼 이름만 추출
category_cols = [c for c in cat_ratio.columns if c.startswith('비율_')]
print(f"\n📊 대분류 피처 ({len(category_cols)}개): {category_cols}")

# ── 스케일링 (비율이지만 GMM의 공분산 추정에 도움됨) ──────────
scaler = StandardScaler()
X = cat_ratio[category_cols].values
X_scaled = scaler.fit_transform(X)

print(f"\n소비자 프로파일 shape: {cat_ratio.shape}")

# ══════════════════════════════════════════════════════════════
# ⑤ BIC / AIC로 최적 K 탐색
# ══════════════════════════════════════════════════════════════
K_range = range(2, 10)
bic_scores, aic_scores = [], []

print("\n[BIC / AIC 탐색]")
for k in K_range:
    gmm = GaussianMixture(n_components=k, random_state=42,
                          covariance_type='full', n_init=10, max_iter=300)
    gmm.fit(X_scaled)
    b, a = gmm.bic(X_scaled), gmm.aic(X_scaled)
    bic_scores.append(b)
    aic_scores.append(a)
    print(f"  k={k:2d} | BIC: {b:10.1f} | AIC: {a:10.1f}")

best_k = list(K_range)[bic_scores.index(min(bic_scores))]
print(f"\n★ BIC 기준 최적 K = {best_k}")

# BIC / AIC 플롯
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, scores, label, color in zip(
        axes,
        [bic_scores, aic_scores],
        ['BIC', 'AIC'],
        ['royalblue', 'tomato']):
    ax.plot(list(K_range), scores, 'o-', color=color, linewidth=2, markersize=7)
    ax.axvline(x=best_k, color='orange', linestyle='--', linewidth=2, label=f'최적 K={best_k}')
    ax.set_title(f'{label} Score (낮을수록 좋음)', fontsize=12)
    ax.set_xlabel('컴포넌트 수 K')
    ax.set_ylabel(label)
    ax.legend()
    ax.grid(linestyle='--', alpha=0.5)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.suptitle('GMM 최적 K 탐색 (대분류 업종 비율 기준)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════
# ⑥ 최적 K로 GMM 학습 & 클러스터 할당
# ══════════════════════════════════════════════════════════════
gmm_final = GaussianMixture(n_components=best_k, random_state=42,
                             covariance_type='full', n_init=10, max_iter=300)
gmm_final.fit(X_scaled)

cat_ratio['cluster'] = gmm_final.predict(X_scaled)

# 소프트 확률
proba_cols = [f'prob_C{i}' for i in range(best_k)]
cat_ratio[proba_cols] = gmm_final.predict_proba(X_scaled).round(3)

print(f"\n[클러스터 분포]")
print(cat_ratio['cluster'].value_counts().sort_index())

# ══════════════════════════════════════════════════════════════
# ⑦ 클러스터별 업종 비율 평균 → 자동 라벨링
# ══════════════════════════════════════════════════════════════
cluster_means = cat_ratio.groupby('cluster')[category_cols].mean()

# 각 클러스터의 지배 업종 (상위 1~2개)
def make_label(row, top_n=2):
    top = row.nlargest(top_n)
    parts = [f"{c.replace('비율_', '')} {v*100:.0f}%" for c, v in top.items()]
    return ' / '.join(parts)

AUTO_LABELS = {i: make_label(cluster_means.loc[i]) for i in range(best_k)}
print("\n[자동 클러스터 라벨]")
for k, v in AUTO_LABELS.items():
    print(f"  Cluster {k}: {v}")

cat_ratio['cluster_name'] = cat_ratio['cluster'].map(AUTO_LABELS)

# ══════════════════════════════════════════════════════════════
# ⑧ 시각화 A: 업종 비율 히트맵
# ══════════════════════════════════════════════════════════════
short_cols = [c.replace('비율_', '') for c in category_cols]

fig, ax = plt.subplots(figsize=(max(10, len(category_cols) * 1.3), best_k * 1.4 + 2))
im = ax.imshow(cluster_means.values, cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.5)

ax.set_xticks(range(len(category_cols)))
ax.set_xticklabels(short_cols, rotation=35, ha='right', fontsize=10)
ax.set_yticks(range(best_k))
ax.set_yticklabels([f'C{i}: {AUTO_LABELS[i]}' for i in range(best_k)], fontsize=9)

for i in range(best_k):
    for j in range(len(category_cols)):
        val = cluster_means.values[i, j]
        ax.text(j, i, f'{val*100:.1f}%',
                ha='center', va='center', fontsize=8,
                color='white' if val > 0.3 else 'black')

plt.colorbar(im, ax=ax, label='업종별 지출 비율')
ax.set_title('클러스터별 업종 지출 비율 히트맵\n(어디에 돈을 쓰는 그룹인가)', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════
# ⑨ 시각화 B: 클러스터별 업종 비율 스택 바차트
# ══════════════════════════════════════════════════════════════
palette = plt.cm.tab10.colors
fig, ax = plt.subplots(figsize=(10, 5))
x = range(best_k)
bottom = np.zeros(best_k)

for j, col in enumerate(category_cols):
    vals = cluster_means[col].values
    ax.bar(x, vals, bottom=bottom,
           label=col.replace('비율_', ''),
           color=palette[j % len(palette)],
           edgecolor='white', linewidth=0.5)
    # 비율이 5% 이상인 경우만 텍스트 표기
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 0.05:
            ax.text(i, b + v / 2, f'{v*100:.0f}%',
                    ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    bottom += vals

ax.set_xticks(list(x))
ax.set_xticklabels([f'C{i}\n{AUTO_LABELS[i]}' for i in range(best_k)], fontsize=8)
ax.set_ylabel('업종 지출 비율')
ax.set_ylim(0, 1.05)
ax.set_title('클러스터별 업종 지출 비율 구성\n(대분류: 어디에 돈 쓰는 사람인가)', fontsize=13)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════
# ⑩ 시각화 C: (age, sex) 그룹이 어느 클러스터에 분포하나
# ══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# C-1: 성별 × 클러스터
sex_cluster = cat_ratio.groupby(['sex', 'cluster']).size().unstack(fill_value=0)
sex_cluster.plot(kind='bar', ax=axes[0], colormap='Set2', edgecolor='white')
axes[0].set_title('성별 × 클러스터 분포', fontsize=12)
axes[0].set_xlabel('성별')
axes[0].set_ylabel('(age, sex) 그룹 수')
axes[0].legend(title='Cluster', bbox_to_anchor=(1.01, 1))
axes[0].tick_params(axis='x', rotation=0)
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# C-2: 연령대 × 클러스터
age_cluster = cat_ratio.groupby(['age', 'cluster']).size().unstack(fill_value=0)
age_cluster.plot(kind='bar', ax=axes[1], colormap='Set2', edgecolor='white')
axes[1].set_title('연령대 × 클러스터 분포', fontsize=12)
axes[1].set_xlabel('연령대')
axes[1].set_ylabel('(age, sex) 그룹 수')
axes[1].legend(title='Cluster', bbox_to_anchor=(1.01, 1))
axes[1].tick_params(axis='x', rotation=0)
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('(age, sex) 그룹 → 업종 기반 클러스터 배정 결과', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════
# ⑪ 클러스터별 상세 요약 출력
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("  클러스터별 업종 소비 요약")
print("="*65)
for cid in range(best_k):
    sub = cat_ratio[cat_ratio['cluster'] == cid]
    mean_row = cluster_means.loc[cid].sort_values(ascending=False)
    print(f"\n📌 Cluster {cid} | {AUTO_LABELS[cid]}")
    print(f"   포함 그룹 수: {len(sub)}개 | 멤버: {sub[['age','sex']].values.tolist()}")
    print("   업종별 평균 비율:")
    for col, val in mean_row.items():
        bar = '█' * int(val * 30)
        print(f"     {col.replace('비율_', ''):12s} {val*100:5.1f}%  {bar}")

# ══════════════════════════════════════════════════════════════
# ⑫ 개인 데이터 예측 함수 (대분류 비율 기반)
# ══════════════════════════════════════════════════════════════
def predict_by_category(individual_df):
    """
    개인 거래 데이터 → 대분류 업종 비율 기반 클러스터 예측
    반환: 클러스터 ID, 라벨, 업종별 지출 비율, 소속 확률
    """
    df_ind = individual_df[individual_df['cnt'] > 0].copy()

    # 대분류별 비율 계산
    cat_amt = df_ind.groupby('card_tpbuz_nm_1')['amt'].sum()
    cat_ratio_dict = (cat_amt / cat_amt.sum()).to_dict()

    feat = {}
    for col in category_cols:
        cat_name = col.replace('비율_', '')
        feat[col] = cat_ratio_dict.get(cat_name, 0.0)

    X_new = np.array([[feat[c] for c in category_cols]])
    X_new_scaled = scaler.transform(X_new)

    cluster_id    = int(gmm_final.predict(X_new_scaled)[0])
    probabilities = gmm_final.predict_proba(X_new_scaled)[0]
    top2_idx      = probabilities.argsort()[::-1][:2]
    top2          = [(f"Cluster {i} ({AUTO_LABELS[i]})", round(probabilities[i]*100, 1))
                     for i in top2_idx]

    return {
        'cluster_id':    cluster_id,
        'cluster_name':  AUTO_LABELS[cluster_id],
        '업종별_지출비율': {c.replace('비율_', ''): f'{v*100:.1f}%' for c, v in feat.items() if v > 0},
        '소속확률_top2': top2,
        '전체확률':      {f'C{i}': round(float(p)*100, 1) for i, p in enumerate(probabilities)},
    }


# ── 데모 실행 ─────────────────────────────────────────────────
demo_df = df_all[(df_all['age'] == 4) & (df_all['sex'] == 'F')]
result  = predict_by_category(demo_df)

print("\n" + "="*60)
print("  대분류 기반 소비 유형 분석 (40대 여성 데모)")
print("="*60)
print(f"  주 소비 유형  : Cluster {result['cluster_id']} - {result['cluster_name']}")
print(f"  소속 확률 1위 : {result['소속확률_top2'][0][0]}  {result['소속확률_top2'][0][1]}%")
print(f"  소속 확률 2위 : {result['소속확률_top2'][1][0]}  {result['소속확률_top2'][1][1]}%")
print(f"  업종별 지출   :")
for cat, ratio in sorted(result['업종별_지출비율'].items(), key=lambda x: -float(x[1].replace('%',''))):
    print(f"    {cat:14s}: {ratio}")
print("="*60)


In [ ]:
import pandas as pd
import numpy as np
import zipfile
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# ─────────────────────────────────────────────────────────────
# ① 데이터 로딩
# ─────────────────────────────────────────────────────────────
data_dir = "data"
zip_files = {
    "2507": "카드소비 데이터_202507.zip",
    "2508": "카드소비 데이터_202508.zip",
    "2509": "카드소비 데이터_202509.zip",
    "2510": "카드소비 데이터_202510.zip",
    "2511": "카드소비 데이터_202511.zip",
    "2512": "카드소비 데이터_202512.zip",
}

all_dfs = []
for month, zipname in zip_files.items():
    zip_path = os.path.join(data_dir, zipname)
    monthly_dfs = []
    with zipfile.ZipFile(zip_path) as zf:
        for filename in sorted(zf.namelist()):
            with zf.open(filename) as f:
                df_city = pd.read_csv(f)
                monthly_dfs.append(df_city)
    combined = pd.concat(monthly_dfs, ignore_index=True)
    combined['month'] = month
    all_dfs.append(combined)
    print(f"✅ {month} 로드 완료 → shape: {combined.shape}")

df_all = pd.concat(all_dfs, ignore_index=True)
print(f"\n✅ 전체 합치기 완료 → shape: {df_all.shape}")

# ─────────────────────────────────────────────────────────────
# ② 전처리 (age, sex 컬럼은 유지하되 피처로는 사용 안 함)
# ─────────────────────────────────────────────────────────────
df_all = df_all[df_all['cnt'] > 0].copy()
df_all['건당가격'] = df_all['amt'] / df_all['cnt']

def remove_outliers_iqr(group):
    Q1 = group['건당가격'].quantile(0.25)
    Q3 = group['건당가격'].quantile(0.75)
    IQR = Q3 - Q1
    return group[(group['건당가격'] >= Q1 - 1.5*IQR) & (group['건당가격'] <= Q3 + 1.5*IQR)]

before = len(df_all)
df_all = df_all.groupby('card_tpbuz_nm_2', group_keys=False).apply(remove_outliers_iqr)
print(f"   이상치 제거: {before:,} → {len(df_all):,}행")

# ══════════════════════════════════════════════════════════════
# ③ 소분류(card_tpbuz_nm_2) 기반 소비 프로파일 생성
#    단위: (age, sex) 조합
#    피처: 각 소분류에서 지출한 "비율" (합계=1)
#    → 나이/성별은 클러스터링에 전혀 사용하지 않음
# ══════════════════════════════════════════════════════════════
print("\n소분류별 피벗 생성 중...")

# (age, sex) × 소분류별 총 결제금액
sub_pivot = (
    df_all.groupby(['age', 'sex', 'card_tpbuz_nm_2'])['amt']
    .sum()
    .reset_index()
    .pivot_table(index=['age', 'sex'], columns='card_tpbuz_nm_2', values='amt', fill_value=0)
)

# 각 (age, sex) 행을 비율화 (합계=1): "이 사람은 어디에 비중을 두나"
sub_ratio = sub_pivot.div(sub_pivot.sum(axis=1), axis=0).round(4)
sub_ratio.columns = [f'비율_{c}' for c in sub_ratio.columns]
sub_ratio = sub_ratio.reset_index()

feature_cols = [c for c in sub_ratio.columns if c.startswith('비율_')]
print(f"📊 소분류 피처 수: {len(feature_cols)}개")
print(f"📊 프로파일 수 (age×sex): {len(sub_ratio)}개")

# ── 스케일링 ──────────────────────────────────────────────────
scaler = StandardScaler()
X = sub_ratio[feature_cols].values
X_scaled = scaler.fit_transform(X)

# ══════════════════════════════════════════════════════════════
# ④ BIC / AIC로 최적 K 탐색
# ══════════════════════════════════════════════════════════════
# → 프로파일 수(22)에 비해 K가 너무 크면 과적합
#   합리적인 상한: min(10, n_samples // 2)
n_samples = len(sub_ratio)
max_k = min(8, n_samples // 2)
K_range = range(2, max_k + 1)
bic_scores, aic_scores = [], []

print(f"\n[BIC / AIC 탐색] (K = 2 ~ {max_k})")
for k in K_range:
    gmm = GaussianMixture(n_components=k, random_state=42,
                          covariance_type='diag',   # diag: 소규모 데이터에 안정적
                          n_init=10, max_iter=300)
    gmm.fit(X_scaled)
    b, a = gmm.bic(X_scaled), gmm.aic(X_scaled)
    bic_scores.append(b)
    aic_scores.append(a)
    print(f"  k={k:2d} | BIC: {b:10.1f} | AIC: {a:10.1f}")

best_k = list(K_range)[bic_scores.index(min(bic_scores))]
print(f"\n★ BIC 기준 최적 K = {best_k}")

# BIC / AIC 플롯
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, scores, label, color in zip(
        axes, [bic_scores, aic_scores], ['BIC', 'AIC'], ['royalblue', 'tomato']):
    ax.plot(list(K_range), scores, 'o-', color=color, linewidth=2, markersize=7)
    ax.axvline(x=best_k, color='orange', linestyle='--', linewidth=2, label=f'최적 K={best_k}')
    ax.set_title(f'{label} Score (낮을수록 좋음)', fontsize=12)
    ax.set_xlabel('컴포넌트 수 K')
    ax.set_ylabel(label)
    ax.legend(); ax.grid(linestyle='--', alpha=0.5)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.suptitle('GMM 최적 K 탐색 (소분류 업종 비율 기준)', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

# ══════════════════════════════════════════════════════════════
# ⑤ 최적 K로 GMM 학습 & 클러스터 할당
# ══════════════════════════════════════════════════════════════
gmm_final = GaussianMixture(n_components=best_k, random_state=42,
                             covariance_type='diag', n_init=10, max_iter=300)
gmm_final.fit(X_scaled)

sub_ratio['cluster'] = gmm_final.predict(X_scaled)
proba_cols = [f'prob_C{i}' for i in range(best_k)]
sub_ratio[proba_cols] = gmm_final.predict_proba(X_scaled).round(3)

print(f"\n[클러스터 분포]")
print(sub_ratio['cluster'].value_counts().sort_index())

# ══════════════════════════════════════════════════════════════
# ⑥ 클러스터 자동 라벨링 (소비 행동만으로 설명)
#    → 상위 2개 소분류 항목으로 이름 생성
# ══════════════════════════════════════════════════════════════
cluster_means = sub_ratio.groupby('cluster')[feature_cols].mean()

def make_behavior_label(row, top_n=2):
    """나이/성별 없이 소비 행동만으로 라벨 생성"""
    top = row.nlargest(top_n)
    parts = [f"{c.replace('비율_', '')}({v*100:.0f}%)" for c, v in top.items()]
    return ' · '.join(parts)

AUTO_LABELS = {i: make_behavior_label(cluster_means.loc[i]) for i in range(best_k)}

print("\n[클러스터 소비 행동 라벨]")
for k, v in AUTO_LABELS.items():
    print(f"  Cluster {k}: {v}")

sub_ratio['cluster_name'] = sub_ratio['cluster'].map(AUTO_LABELS)

# ══════════════════════════════════════════════════════════════
# ⑦ 시각화 A: 상위 소분류 히트맵 (상위 N개만 표시)
# ══════════════════════════════════════════════════════════════
# 전체 평균 기준 상위 20개 소분류만 골라서 보기 좋게
top_n_cols = cluster_means.mean().nlargest(20).index.tolist()
top_n_labels = [c.replace('비율_', '') for c in top_n_cols]
heatmap_data = cluster_means[top_n_cols]

fig, ax = plt.subplots(figsize=(18, best_k * 1.5 + 2))
im = ax.imshow(heatmap_data.values, cmap='YlOrRd', aspect='auto')

ax.set_xticks(range(len(top_n_cols)))
ax.set_xticklabels(top_n_labels, rotation=40, ha='right', fontsize=9)
ax.set_yticks(range(best_k))
ax.set_yticklabels([f'C{i}: {AUTO_LABELS[i]}' for i in range(best_k)], fontsize=9)

for i in range(best_k):
    for j in range(len(top_n_cols)):
        val = heatmap_data.values[i, j]
        ax.text(j, i, f'{val*100:.1f}%',
                ha='center', va='center', fontsize=7.5,
                color='white' if val > 0.25 else 'black')

plt.colorbar(im, ax=ax, label='지출 비율')
ax.set_title('클러스터별 소비 행동 히트맵 (상위 20개 소분류)\n어디에 돈을 쓰는 소비자인가', fontsize=13, pad=12)
plt.tight_layout(); plt.show()

# ══════════════════════════════════════════════════════════════
# ⑧ 시각화 B: 클러스터별 소분류 Top5 바차트 (클러스터당 1개)
# ══════════════════════════════════════════════════════════════
n_cols = min(best_k, 4)
n_rows = (best_k + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
axes = np.array(axes).flatten()

colors = plt.cm.tab10.colors

for cid in range(best_k):
    ax = axes[cid]
    top5 = cluster_means.loc[cid].nlargest(8)
    labels = [c.replace('비율_', '') for c in top5.index]
    vals   = top5.values * 100

    bars = ax.barh(labels[::-1], vals[::-1],
                   color=[colors[cid % len(colors)]] * len(labels),
                   edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, vals[::-1]):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=9, fontweight='bold')

    ax.set_title(f'Cluster {cid}\n{AUTO_LABELS[cid]}', fontsize=10, fontweight='bold')
    ax.set_xlabel('지출 비율 (%)')
    ax.set_xlim(0, max(vals) * 1.2 + 5)
    ax.grid(axis='x', linestyle='--', alpha=0.4)
    ax.spines[['top', 'right']].set_visible(False)

# 빈 subplot 숨기기
for i in range(best_k, len(axes)):
    axes[i].set_visible(False)

plt.suptitle('클러스터별 주요 소비 항목 Top8\n(나이·성별 무관한 순수 소비 행동 유형)', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

# ══════════════════════════════════════════════════════════════
# ⑨ 클러스터별 소비 행동 요약 출력 (나이/성별 제거)
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("  소비 행동 유형 클러스터 요약")
print("  (나이·성별 무관 — 순수히 어디에 돈 쓰는 사람인가)")
print("="*65)

for cid in range(best_k):
    top_items = cluster_means.loc[cid].nlargest(5)
    sub = sub_ratio[sub_ratio['cluster'] == cid]
    n_members = len(sub)

    print(f"\n📌 Cluster {cid} | '{AUTO_LABELS[cid]}' 소비형")
    print(f"   포함 프로파일 수: {n_members}개")
    print(f"   주요 소비 항목:")
    for col, val in top_items.items():
        item = col.replace('비율_', '')
        bar  = '█' * int(val * 40)
        print(f"     {item:16s}  {val*100:5.1f}%  {bar}")

# ══════════════════════════════════════════════════════════════
# ⑩ 개인 데이터 예측 함수
#    → 반환값에 나이/성별 없음: 순수 소비 행동만 서술
# ══════════════════════════════════════════════════════════════
def predict_spending_type(individual_df):
    """
    개인 거래 데이터 → 소분류 업종 비율 기반 소비 유형 예측
    나이·성별은 전혀 사용하지 않음
    """
    df_ind = individual_df[individual_df['cnt'] > 0].copy()

    # 소분류별 비율
    cat_amt = df_ind.groupby('card_tpbuz_nm_2')['amt'].sum()
    cat_ratio_dict = (cat_amt / cat_amt.sum()).to_dict()

    feat = {col: cat_ratio_dict.get(col.replace('비율_', ''), 0.0) for col in feature_cols}
    X_new        = np.array([[feat[c] for c in feature_cols]])
    X_new_scaled = scaler.transform(X_new)

    cluster_id    = int(gmm_final.predict(X_new_scaled)[0])
    probabilities = gmm_final.predict_proba(X_new_scaled)[0]
    top2_idx      = probabilities.argsort()[::-1][:2]

    # 상위 소비 항목 (비율 기준)
    top_items = sorted(
        [(col.replace('비율_', ''), v) for col, v in feat.items() if v > 0.01],
        key=lambda x: -x[1]
    )[:5]

    return {
        'cluster_id':    cluster_id,
        'cluster_name':  AUTO_LABELS[cluster_id],  # 나이/성별 없는 순수 행동 라벨
        '소비_유형_설명': f"{AUTO_LABELS[cluster_id]} 소비자",
        '주요_소비_항목': {item: f'{v*100:.1f}%' for item, v in top_items},
        '소속확률_top2': [
            (f"C{i} ({AUTO_LABELS[i]})", round(float(probabilities[i])*100, 1))
            for i in top2_idx
        ],
    }


# ── 데모 실행 ─────────────────────────────────────────────────
demo_df = df_all[(df_all['age'] == 4) & (df_all['sex'] == 'F')]
result  = predict_spending_type(demo_df)

print("\n" + "="*60)
print("  소비 행동 유형 분석 결과 (데모)")
print("="*60)
print(f"  소비 유형   : {result['소비_유형_설명']}")
print(f"  확률 1위    : {result['소속확률_top2'][0][0]}  {result['소속확률_top2'][0][1]}%")
print(f"  확률 2위    : {result['소속확률_top2'][1][0]}  {result['소속확률_top2'][1][1]}%")
print(f"  주요 소비   :")
for item, ratio in result['주요_소비_항목'].items():
    print(f"    {item:18s}: {ratio}")
print("="*60)


In [ ]:
import pandas as pd
import numpy as np
import zipfile
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# ─────────────────────────────────────────────────────────────
# ① 데이터 로딩
# ─────────────────────────────────────────────────────────────
data_dir = "data"
zip_files = {
    "2507": "카드소비 데이터_202507.zip",
    "2508": "카드소비 데이터_202508.zip",
    "2509": "카드소비 데이터_202509.zip",
    "2510": "카드소비 데이터_202510.zip",
    "2511": "카드소비 데이터_202511.zip",
    "2512": "카드소비 데이터_202512.zip",
}

all_dfs = []
for month, zipname in zip_files.items():
    zip_path = os.path.join(data_dir, zipname)
    monthly_dfs = []
    with zipfile.ZipFile(zip_path) as zf:
        for filename in sorted(zf.namelist()):
            with zf.open(filename) as f:
                df_city = pd.read_csv(f)
                monthly_dfs.append(df_city)
    combined = pd.concat(monthly_dfs, ignore_index=True)
    combined['month'] = month
    all_dfs.append(combined)
    print(f"✅ {month} 로드 완료 → shape: {combined.shape}")

df_all = pd.concat(all_dfs, ignore_index=True)
print(f"\n✅ 전체 합치기 완료 → shape: {df_all.shape}")

# ─────────────────────────────────────────────────────────────
# ② 전처리 (age, sex 컬럼은 유지하되 피처로는 사용 안 함)
# ─────────────────────────────────────────────────────────────
df_all = df_all[df_all['cnt'] > 0].copy()
df_all['건당가격'] = df_all['amt'] / df_all['cnt']

def remove_outliers_iqr(group):
    Q1 = group['건당가격'].quantile(0.25)
    Q3 = group['건당가격'].quantile(0.75)
    IQR = Q3 - Q1
    return group[(group['건당가격'] >= Q1 - 1.5*IQR) & (group['건당가격'] <= Q3 + 1.5*IQR)]

before = len(df_all)
df_all = df_all.groupby('card_tpbuz_nm_2', group_keys=False).apply(remove_outliers_iqr)
print(f"   이상치 제거: {before:,} → {len(df_all):,}행")

# ══════════════════════════════════════════════════════════════
# ③ 소분류(card_tpbuz_nm_2) 기반 소비 프로파일 생성
#    단위: (age, sex) 조합
#    피처: 각 소분류에서 지출한 "비율" (합계=1)
#    → 나이/성별은 클러스터링에 전혀 사용하지 않음
# ══════════════════════════════════════════════════════════════
print("\n소분류별 피벗 생성 중...")

# (age, sex) × 소분류별 총 결제금액
sub_pivot = (
    df_all.groupby(['age', 'sex', 'card_tpbuz_nm_2'])['amt']
    .sum()
    .reset_index()
    .pivot_table(index=['age', 'sex'], columns='card_tpbuz_nm_2', values='amt', fill_value=0)
)

# 각 (age, sex) 행을 비율화 (합계=1): "이 사람은 어디에 비중을 두나"
sub_ratio = sub_pivot.div(sub_pivot.sum(axis=1), axis=0).round(4)
sub_ratio.columns = [f'비율_{c}' for c in sub_ratio.columns]
sub_ratio = sub_ratio.reset_index()

# ── 종합소매점 제외 ──────────────────────────────────────────
# 종합소매점은 마트·편의점·쿠팡 등을 모두 포함하는 광범위 카테고리.
# 모든 소비자가 비슷하게 이용하기 때문에 클러스터 구분력이 거의 없음.
# 제외해야 '한식 애호', '커피샵', '병원' 같은 실제 소비 성향 차이가 드러남.
EXCLUDE_ITEMS = ['종합소매점']
exclude_cols  = [f'비율_{item}' for item in EXCLUDE_ITEMS]

feature_cols = [
    c for c in sub_ratio.columns
    if c.startswith('비율_') and c not in exclude_cols
]
print(f"📊 소분류 피처 수: {len(feature_cols)}개 (종합소매점 제외)")
print(f"📊 프로파일 수 (age×sex): {len(sub_ratio)}개")

# ── 스케일링 ──────────────────────────────────────────────────
scaler = StandardScaler()
X = sub_ratio[feature_cols].values
X_scaled = scaler.fit_transform(X)

# ══════════════════════════════════════════════════════════════
# ④ BIC / AIC로 최적 K 탐색
# ══════════════════════════════════════════════════════════════
# → 프로파일 수(22)에 비해 K가 너무 크면 과적합
#   합리적인 상한: min(10, n_samples // 2)
n_samples = len(sub_ratio)
max_k = min(8, n_samples // 2)
K_range = range(2, max_k + 1)
bic_scores, aic_scores = [], []

print(f"\n[BIC / AIC 탐색] (K = 2 ~ {max_k})")
for k in K_range:
    gmm = GaussianMixture(n_components=k, random_state=42,
                          covariance_type='diag',   # diag: 소규모 데이터에 안정적
                          n_init=10, max_iter=300)
    gmm.fit(X_scaled)
    b, a = gmm.bic(X_scaled), gmm.aic(X_scaled)
    bic_scores.append(b)
    aic_scores.append(a)
    print(f"  k={k:2d} | BIC: {b:10.1f} | AIC: {a:10.1f}")

best_k = list(K_range)[bic_scores.index(min(bic_scores))]
print(f"\n★ BIC 기준 최적 K = {best_k}")

# BIC / AIC 플롯
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, scores, label, color in zip(
        axes, [bic_scores, aic_scores], ['BIC', 'AIC'], ['royalblue', 'tomato']):
    ax.plot(list(K_range), scores, 'o-', color=color, linewidth=2, markersize=7)
    ax.axvline(x=best_k, color='orange', linestyle='--', linewidth=2, label=f'최적 K={best_k}')
    ax.set_title(f'{label} Score (낮을수록 좋음)', fontsize=12)
    ax.set_xlabel('컴포넌트 수 K')
    ax.set_ylabel(label)
    ax.legend(); ax.grid(linestyle='--', alpha=0.5)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.suptitle('GMM 최적 K 탐색 (소분류 업종 비율 기준)', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

# ══════════════════════════════════════════════════════════════
# ⑤ 최적 K로 GMM 학습 & 클러스터 할당
# ══════════════════════════════════════════════════════════════
gmm_final = GaussianMixture(n_components=best_k, random_state=42,
                             covariance_type='diag', n_init=10, max_iter=300)
gmm_final.fit(X_scaled)

sub_ratio['cluster'] = gmm_final.predict(X_scaled)
proba_cols = [f'prob_C{i}' for i in range(best_k)]
sub_ratio[proba_cols] = gmm_final.predict_proba(X_scaled).round(3)

print(f"\n[클러스터 분포]")
print(sub_ratio['cluster'].value_counts().sort_index())

# ══════════════════════════════════════════════════════════════
# ⑥ 클러스터 자동 라벨링 (소비 행동만으로 설명)
#    → 상위 2개 소분류 항목으로 이름 생성
# ══════════════════════════════════════════════════════════════
cluster_means = sub_ratio.groupby('cluster')[feature_cols].mean()

def make_behavior_label(row, top_n=2):
    """나이/성별 없이 소비 행동만으로 라벨 생성"""
    top = row.nlargest(top_n)
    parts = [f"{c.replace('비율_', '')}({v*100:.0f}%)" for c, v in top.items()]
    return ' · '.join(parts)

AUTO_LABELS = {i: make_behavior_label(cluster_means.loc[i]) for i in range(best_k)}

print("\n[클러스터 소비 행동 라벨]")
for k, v in AUTO_LABELS.items():
    print(f"  Cluster {k}: {v}")

sub_ratio['cluster_name'] = sub_ratio['cluster'].map(AUTO_LABELS)

# ══════════════════════════════════════════════════════════════
# ⑦ 시각화 A: 상위 소분류 히트맵 (상위 N개만 표시)
# ══════════════════════════════════════════════════════════════
# 전체 평균 기준 상위 20개 소분류만 골라서 보기 좋게
top_n_cols = cluster_means.mean().nlargest(20).index.tolist()
top_n_labels = [c.replace('비율_', '') for c in top_n_cols]
heatmap_data = cluster_means[top_n_cols]

fig, ax = plt.subplots(figsize=(18, best_k * 1.5 + 2))
im = ax.imshow(heatmap_data.values, cmap='YlOrRd', aspect='auto')

ax.set_xticks(range(len(top_n_cols)))
ax.set_xticklabels(top_n_labels, rotation=40, ha='right', fontsize=9)
ax.set_yticks(range(best_k))
ax.set_yticklabels([f'C{i}: {AUTO_LABELS[i]}' for i in range(best_k)], fontsize=9)

for i in range(best_k):
    for j in range(len(top_n_cols)):
        val = heatmap_data.values[i, j]
        ax.text(j, i, f'{val*100:.1f}%',
                ha='center', va='center', fontsize=7.5,
                color='white' if val > 0.25 else 'black')

plt.colorbar(im, ax=ax, label='지출 비율')
ax.set_title('클러스터별 소비 행동 히트맵 (상위 20개 소분류)\n어디에 돈을 쓰는 소비자인가', fontsize=13, pad=12)
plt.tight_layout(); plt.show()

# ══════════════════════════════════════════════════════════════
# ⑧ 시각화 B: 클러스터별 소분류 Top5 바차트 (클러스터당 1개)
# ══════════════════════════════════════════════════════════════
n_cols = min(best_k, 4)
n_rows = (best_k + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
axes = np.array(axes).flatten()

colors = plt.cm.tab10.colors

for cid in range(best_k):
    ax = axes[cid]
    top5 = cluster_means.loc[cid].nlargest(8)
    labels = [c.replace('비율_', '') for c in top5.index]
    vals   = top5.values * 100

    bars = ax.barh(labels[::-1], vals[::-1],
                   color=[colors[cid % len(colors)]] * len(labels),
                   edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, vals[::-1]):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=9, fontweight='bold')

    ax.set_title(f'Cluster {cid}\n{AUTO_LABELS[cid]}', fontsize=10, fontweight='bold')
    ax.set_xlabel('지출 비율 (%)')
    ax.set_xlim(0, max(vals) * 1.2 + 5)
    ax.grid(axis='x', linestyle='--', alpha=0.4)
    ax.spines[['top', 'right']].set_visible(False)

# 빈 subplot 숨기기
for i in range(best_k, len(axes)):
    axes[i].set_visible(False)

plt.suptitle('클러스터별 주요 소비 항목 Top8\n(나이·성별 무관한 순수 소비 행동 유형)', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

# ══════════════════════════════════════════════════════════════
# ⑨ 클러스터별 소비 행동 요약 출력 (나이/성별 제거)
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("  소비 행동 유형 클러스터 요약")
print("  (나이·성별 무관 — 순수히 어디에 돈 쓰는 사람인가)")
print("="*65)

for cid in range(best_k):
    top_items = cluster_means.loc[cid].nlargest(5)
    sub = sub_ratio[sub_ratio['cluster'] == cid]
    n_members = len(sub)

    print(f"\n📌 Cluster {cid} | '{AUTO_LABELS[cid]}' 소비형")
    print(f"   포함 프로파일 수: {n_members}개")
    print(f"   주요 소비 항목:")
    for col, val in top_items.items():
        item = col.replace('비율_', '')
        bar  = '█' * int(val * 40)
        print(f"     {item:16s}  {val*100:5.1f}%  {bar}")

# ══════════════════════════════════════════════════════════════
# ⑩ 개인 데이터 예측 함수
#    → 반환값에 나이/성별 없음: 순수 소비 행동만 서술
# ══════════════════════════════════════════════════════════════
def predict_spending_type(individual_df):
    """
    개인 거래 데이터 → 소분류 업종 비율 기반 소비 유형 예측
    나이·성별은 전혀 사용하지 않음
    """
    df_ind = individual_df[individual_df['cnt'] > 0].copy()

    # 소분류별 비율
    cat_amt = df_ind.groupby('card_tpbuz_nm_2')['amt'].sum()
    cat_ratio_dict = (cat_amt / cat_amt.sum()).to_dict()

    feat = {col: cat_ratio_dict.get(col.replace('비율_', ''), 0.0) for col in feature_cols}
    X_new        = np.array([[feat[c] for c in feature_cols]])
    X_new_scaled = scaler.transform(X_new)

    cluster_id    = int(gmm_final.predict(X_new_scaled)[0])
    probabilities = gmm_final.predict_proba(X_new_scaled)[0]
    top2_idx      = probabilities.argsort()[::-1][:2]

    # 상위 소비 항목 (비율 기준)
    top_items = sorted(
        [(col.replace('비율_', ''), v) for col, v in feat.items() if v > 0.01],
        key=lambda x: -x[1]
    )[:5]

    return {
        'cluster_id':    cluster_id,
        'cluster_name':  AUTO_LABELS[cluster_id],  # 나이/성별 없는 순수 행동 라벨
        '소비_유형_설명': f"{AUTO_LABELS[cluster_id]} 소비자",
        '주요_소비_항목': {item: f'{v*100:.1f}%' for item, v in top_items},
        '소속확률_top2': [
            (f"C{i} ({AUTO_LABELS[i]})", round(float(probabilities[i])*100, 1))
            for i in top2_idx
        ],
    }


# ── 데모 실행 ─────────────────────────────────────────────────
demo_df = df_all[(df_all['age'] == 4) & (df_all['sex'] == 'F')]
result  = predict_spending_type(demo_df)

print("\n" + "="*60)
print("  소비 행동 유형 분석 결과 (데모)")
print("="*60)
print(f"  소비 유형   : {result['소비_유형_설명']}")
print(f"  확률 1위    : {result['소속확률_top2'][0][0]}  {result['소속확률_top2'][0][1]}%")
print(f"  확률 2위    : {result['소속확률_top2'][1][0]}  {result['소속확률_top2'][1][1]}%")
print(f"  주요 소비   :")
for item, ratio in result['주요_소비_항목'].items():
    print(f"    {item:18s}: {ratio}")
print("="*60)


In [ ]:
import pandas as pd
import numpy as np
import zipfile
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# ─────────────────────────────────────────────────────────────
# ① 데이터 로딩
# ─────────────────────────────────────────────────────────────
data_dir = "data"
zip_files = {
    "2507": "카드소비 데이터_202507.zip",
    "2508": "카드소비 데이터_202508.zip",
    "2509": "카드소비 데이터_202509.zip",
    "2510": "카드소비 데이터_202510.zip",
    "2511": "카드소비 데이터_202511.zip",
    "2512": "카드소비 데이터_202512.zip",
}

all_dfs = []
for month, zipname in zip_files.items():
    zip_path = os.path.join(data_dir, zipname)
    monthly_dfs = []
    with zipfile.ZipFile(zip_path) as zf:
        for filename in sorted(zf.namelist()):
            with zf.open(filename) as f:
                df_city = pd.read_csv(f)
                monthly_dfs.append(df_city)
    combined = pd.concat(monthly_dfs, ignore_index=True)
    combined['month'] = month
    all_dfs.append(combined)
    print(f"✅ {month} 로드 완료 → shape: {combined.shape}")

df_all = pd.concat(all_dfs, ignore_index=True)
print(f"\n✅ 전체 합치기 완료 → shape: {df_all.shape}")

# ─────────────────────────────────────────────────────────────
# ② 전처리 (age, sex 컬럼은 유지하되 피처로는 사용 안 함)
# ─────────────────────────────────────────────────────────────
df_all = df_all[df_all['cnt'] > 0].copy()
df_all['건당가격'] = df_all['amt'] / df_all['cnt']

def remove_outliers_iqr(group):
    Q1 = group['건당가격'].quantile(0.25)
    Q3 = group['건당가격'].quantile(0.75)
    IQR = Q3 - Q1
    return group[(group['건당가격'] >= Q1 - 1.5*IQR) & (group['건당가격'] <= Q3 + 1.5*IQR)]

before = len(df_all)
df_all = df_all.groupby('card_tpbuz_nm_2', group_keys=False).apply(remove_outliers_iqr)
print(f"   이상치 제거: {before:,} → {len(df_all):,}행")

# ══════════════════════════════════════════════════════════════
# ③ 소분류 카테고리 정제 및 통폐합 (차원 축소)
# ══════════════════════════════════════════════════════════════
CATEGORY_MAP = {
    # ── 제외할 쓰레기/공통/특색없는 항목 ──
    '종합소매점': '제외',
    '기타결제': '제외', '기타용품': '제외', '기타의료': '제외', '기타교육': '제외',
    '공공기관': '제외', '기업': '제외', '단체': '제외', '종교': '제외',
    '회비/공과금': '제외', '휴게소/대형업체': '제외', '가례서비스': '제외',
    '제조/도매': '제외', '전문서비스': '제외', '광고/인쇄/인화': '제외',
    '금융상품/서비스': '제외', '무점포서비스': '제외', '보안/운송': '제외', '부동산': '제외',
    
    # ── 통폐합할 카테고리 ──
    '일반병원': '병원/의료', '종합병원': '병원/의료', '특화병원': '병원/의료',
    '차량관리/부품': '자동차/유지비', '차량관리/서비스': '자동차/유지비', '차량판매': '자동차/유지비', '연료판매': '자동차/유지비',
    '유아교육': '교육/학원', '입시학원': '교육/학원', '외국어학원': '교육/학원', 
    '기술/직업교육학원': '교육/학원', '예체능계학원': '교육/학원', '독서실/고시원': '교육/학원',
    '유흥주점': '주점', '간이주점': '주점',
    '고기요리': '육류/회식', '닭/오리요리': '육류/회식',
    '일식/수산물': '기타외식', '별식/퓨전요리': '기타외식', '양식': '기타외식', '중식': '기타외식', '부페': '기타외식',
    '사우나/휴게시설': '건강/뷰티/마사지', '미용서비스': '건강/뷰티/마사지', '요가/단전/마사지': '건강/뷰티/마사지',
    # 그 외 나머지는 기존 이름 그대로 유지 (ex. 한식, 커피/음료, 편의점 등)
}

df_all['refined_category'] = df_all['card_tpbuz_nm_2'].map(lambda x: CATEGORY_MAP.get(x, x))
df_all = df_all[df_all['refined_category'] != '제외'].copy()

print(f"   카테고리 통폐합 후: {len(df_all):,}행")

# ══════════════════════════════════════════════════════════════
# ④ 정제된 소분류 기반 소비 프로파일 생성
#    단위: (age, sex) 조합 / 피처: 합병된 카테고리 지출 "비율"
# ══════════════════════════════════════════════════════════════
print("\n카테고리 비율 피벗 생성 중...")

# (age, sex) × 정제된 카테고리별 총 결제금액
sub_pivot = (
    df_all.groupby(['age', 'sex', 'refined_category'])['amt']
    .sum()
    .reset_index()
    .pivot_table(index=['age', 'sex'], columns='refined_category', values='amt', fill_value=0)
)

# 각 (age, sex) 행을 비율화 (합계=1): "이 사람은 어디에 비중을 두나"
sub_ratio = sub_pivot.div(sub_pivot.sum(axis=1), axis=0).round(4)
sub_ratio.columns = [f'비율_{c}' for c in sub_ratio.columns]
sub_ratio = sub_ratio.reset_index()

feature_cols = [c for c in sub_ratio.columns if c.startswith('비율_')]
print(f"📊 정제된 피처 수: {len(feature_cols)}개")
print(f"📊 프로파일 수 (age×sex): {len(sub_ratio)}개")

# ── 스케일링 ──────────────────────────────────────────────────
scaler = StandardScaler()
X = sub_ratio[feature_cols].values
X_scaled = scaler.fit_transform(X)

# ══════════════════════════════════════════════════════════════
# ⑤ BIC / AIC로 최적 K 탐색
# ══════════════════════════════════════════════════════════════
# → 프로파일 수(22)에 비해 K가 너무 크면 과적합
#   합리적인 상한: min(10, n_samples // 2)
n_samples = len(sub_ratio)
max_k = min(8, n_samples // 2)
K_range = range(2, max_k + 1)
bic_scores, aic_scores = [], []

print(f"\n[BIC / AIC 탐색] (K = 2 ~ {max_k})")
for k in K_range:
    gmm = GaussianMixture(n_components=k, random_state=42,
                          covariance_type='diag',   # diag: 소규모 데이터에 안정적
                          n_init=10, max_iter=300)
    gmm.fit(X_scaled)
    b, a = gmm.bic(X_scaled), gmm.aic(X_scaled)
    bic_scores.append(b)
    aic_scores.append(a)
    print(f"  k={k:2d} | BIC: {b:10.1f} | AIC: {a:10.1f}")

best_k = list(K_range)[bic_scores.index(min(bic_scores))]
print(f"\n★ BIC 기준 최적 K = {best_k}")

# BIC / AIC 플롯
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, scores, label, color in zip(
        axes, [bic_scores, aic_scores], ['BIC', 'AIC'], ['royalblue', 'tomato']):
    ax.plot(list(K_range), scores, 'o-', color=color, linewidth=2, markersize=7)
    ax.axvline(x=best_k, color='orange', linestyle='--', linewidth=2, label=f'최적 K={best_k}')
    ax.set_title(f'{label} Score (낮을수록 좋음)', fontsize=12)
    ax.set_xlabel('컴포넌트 수 K')
    ax.set_ylabel(label)
    ax.legend(); ax.grid(linestyle='--', alpha=0.5)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.suptitle('GMM 최적 K 탐색 (소분류 업종 비율 기준)', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

# ══════════════════════════════════════════════════════════════
# ⑥ 최적 K로 GMM 학습 & 클러스터 할당
# ══════════════════════════════════════════════════════════════
gmm_final = GaussianMixture(n_components=best_k, random_state=42,
                             covariance_type='diag', n_init=10, max_iter=300)
gmm_final.fit(X_scaled)

sub_ratio['cluster'] = gmm_final.predict(X_scaled)
proba_cols = [f'prob_C{i}' for i in range(best_k)]
sub_ratio[proba_cols] = gmm_final.predict_proba(X_scaled).round(3)

print(f"\n[클러스터 분포]")
print(sub_ratio['cluster'].value_counts().sort_index())

# ══════════════════════════════════════════════════════════════
# ⑦ 클러스터 자동 라벨링 (소비 행동만으로 설명)
#    → 상위 2개 소분류 항목으로 이름 생성
# ══════════════════════════════════════════════════════════════
cluster_means = sub_ratio.groupby('cluster')[feature_cols].mean()

def make_behavior_label(row, top_n=2):
    """나이/성별 없이 소비 행동만으로 라벨 생성"""
    top = row.nlargest(top_n)
    parts = [f"{c.replace('비율_', '')}({v*100:.0f}%)" for c, v in top.items()]
    return ' · '.join(parts)

AUTO_LABELS = {i: make_behavior_label(cluster_means.loc[i]) for i in range(best_k)}

print("\n[클러스터 소비 행동 라벨]")
for k, v in AUTO_LABELS.items():
    print(f"  Cluster {k}: {v}")

sub_ratio['cluster_name'] = sub_ratio['cluster'].map(AUTO_LABELS)

# ══════════════════════════════════════════════════════════════
# ⑧ 시각화 A: 상위 카테고리 히트맵 (상위 N개만 표시)
# ══════════════════════════════════════════════════════════════
# 전체 평균 기준 상위 20개 소분류만 골라서 보기 좋게
top_n_cols = cluster_means.mean().nlargest(20).index.tolist()
top_n_labels = [c.replace('비율_', '') for c in top_n_cols]
heatmap_data = cluster_means[top_n_cols]

fig, ax = plt.subplots(figsize=(18, best_k * 1.5 + 2))
im = ax.imshow(heatmap_data.values, cmap='YlOrRd', aspect='auto')

ax.set_xticks(range(len(top_n_cols)))
ax.set_xticklabels(top_n_labels, rotation=40, ha='right', fontsize=9)
ax.set_yticks(range(best_k))
ax.set_yticklabels([f'C{i}: {AUTO_LABELS[i]}' for i in range(best_k)], fontsize=9)

for i in range(best_k):
    for j in range(len(top_n_cols)):
        val = heatmap_data.values[i, j]
        ax.text(j, i, f'{val*100:.1f}%',
                ha='center', va='center', fontsize=7.5,
                color='white' if val > 0.25 else 'black')

plt.colorbar(im, ax=ax, label='지출 비율')
ax.set_title('클러스터별 소비 행동 히트맵 (상위 20개 소분류)\n어디에 돈을 쓰는 소비자인가', fontsize=13, pad=12)
plt.tight_layout(); plt.show()

# ══════════════════════════════════════════════════════════════
# ⑨ 시각화 B: 클러스터별 핵심 카테고리 Top5 바차트 (클러스터당 1개)
# ══════════════════════════════════════════════════════════════
n_cols = min(best_k, 4)
n_rows = (best_k + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
axes = np.array(axes).flatten()

colors = plt.cm.tab10.colors

for cid in range(best_k):
    ax = axes[cid]
    top5 = cluster_means.loc[cid].nlargest(8)
    labels = [c.replace('비율_', '') for c in top5.index]
    vals   = top5.values * 100

    bars = ax.barh(labels[::-1], vals[::-1],
                   color=[colors[cid % len(colors)]] * len(labels),
                   edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, vals[::-1]):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=9, fontweight='bold')

    ax.set_title(f'Cluster {cid}\n{AUTO_LABELS[cid]}', fontsize=10, fontweight='bold')
    ax.set_xlabel('지출 비율 (%)')
    ax.set_xlim(0, max(vals) * 1.2 + 5)
    ax.grid(axis='x', linestyle='--', alpha=0.4)
    ax.spines[['top', 'right']].set_visible(False)

# 빈 subplot 숨기기
for i in range(best_k, len(axes)):
    axes[i].set_visible(False)

plt.suptitle('클러스터별 주요 소비 항목 Top8\n(나이·성별 무관한 순수 소비 행동 유형)', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

# ══════════════════════════════════════════════════════════════
# ⑩ 클러스터별 소비 행동 요약 출력 (나이/성별 제거)
# ══════════════════════════════════════════════════════════════
print("\n" + "="*65)
print("  소비 행동 유형 클러스터 요약")
print("  (나이·성별 무관 — 순수히 어디에 돈 쓰는 사람인가)")
print("="*65)

for cid in range(best_k):
    top_items = cluster_means.loc[cid].nlargest(5)
    sub = sub_ratio[sub_ratio['cluster'] == cid]
    n_members = len(sub)

    print(f"\n📌 Cluster {cid} | '{AUTO_LABELS[cid]}' 소비형")
    print(f"   포함 프로파일 수: {n_members}개")
    print(f"   주요 소비 항목:")
    for col, val in top_items.items():
        item = col.replace('비율_', '')
        bar  = '█' * int(val * 40)
        print(f"     {item:16s}  {val*100:5.1f}%  {bar}")

# ══════════════════════════════════════════════════════════════
# ⑪ 개인 데이터 예측 함수
#    → 반환값에 나이/성별 없음: 순수 소비 행동만 서술
# ══════════════════════════════════════════════════════════════
def predict_spending_type(individual_df):
    """
    개인 거래 데이터 → 소분류 업종 비율 기반 소비 유형 예측
    나이·성별은 전혀 사용하지 않음
    """
    df_ind = individual_df[individual_df['cnt'] > 0].copy()

    # 정제된 분류별 비율
    df_ind['refined_category'] = df_ind['card_tpbuz_nm_2'].map(lambda x: CATEGORY_MAP.get(x, x))
    df_ind = df_ind[df_ind['refined_category'] != '제외'].copy()
    
    cat_amt = df_ind.groupby('refined_category')['amt'].sum()
    cat_ratio_dict = (cat_amt / cat_amt.sum()).to_dict()

    feat = {col: cat_ratio_dict.get(col.replace('비율_', ''), 0.0) for col in feature_cols}
    X_new        = np.array([[feat[c] for c in feature_cols]])
    X_new_scaled = scaler.transform(X_new)

    cluster_id    = int(gmm_final.predict(X_new_scaled)[0])
    probabilities = gmm_final.predict_proba(X_new_scaled)[0]
    top2_idx      = probabilities.argsort()[::-1][:2]

    # 상위 소비 항목 (비율 기준)
    top_items = sorted(
        [(col.replace('비율_', ''), v) for col, v in feat.items() if v > 0.01],
        key=lambda x: -x[1]
    )[:5]

    return {
        'cluster_id':    cluster_id,
        'cluster_name':  AUTO_LABELS[cluster_id],  # 나이/성별 없는 순수 행동 라벨
        '소비_유형_설명': f"{AUTO_LABELS[cluster_id]} 소비자",
        '주요_소비_항목': {item: f'{v*100:.1f}%' for item, v in top_items},
        '소속확률_top2': [
            (f"C{i} ({AUTO_LABELS[i]})", round(float(probabilities[i])*100, 1))
            for i in top2_idx
        ],
    }


# ── 데모 실행 ─────────────────────────────────────────────────
demo_df = df_all[(df_all['age'] == 4) & (df_all['sex'] == 'F')]
result  = predict_spending_type(demo_df)

print("\n" + "="*60)
print("  소비 행동 유형 분석 결과 (데모)")
print("="*60)
print(f"  소비 유형   : {result['소비_유형_설명']}")
print(f"  확률 1위    : {result['소속확률_top2'][0][0]}  {result['소속확률_top2'][0][1]}%")
print(f"  확률 2위    : {result['소속확률_top2'][1][0]}  {result['소속확률_top2'][1][1]}%")
print(f"  주요 소비   :")
for item, ratio in result['주요_소비_항목'].items():
    print(f"    {item:18s}: {ratio}")
print("="*60)
